#### Retrieve PDFs and Merge PDFs from BUCKET to BUCKET 


In [ ]:
import fitz  # PyMuPDF for PDF manipulation
import pymongo
import boto3
from bson import ObjectId
from io import BytesIO

# MongoDB Connection
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["jef-rag"]  # Your database name
collection = db["pdf_storage"]  # Your collection name

# AWS S3 Configuration
s3_client = boto3.client(
    "s3",
    aws_access_key_id="YOUR_ACCESS_KEY",
    aws_secret_access_key="YOUR_SECRET_KEY",
    region_name="ap-southeast-1"  # Change to your region
)
bucket_name = "your-bucket-name"
folder_path = "merged_pdfs/"  # Change to your target folder

# Define keys (MongoDB ObjectIds or another unique field)
keys = ["65b5a1f3e1a2b3c4d5e6f7g8", "65b5a1f3e1a2b3c4d5e6f7g9", "65b5a1f3e1a2b3c4d5e6f7h0"]  # Replace with actual IDs

# Function to retrieve PDFs from MongoDB
def retrieve_pdf(key):
    document = collection.find_one({"_id": ObjectId(key)})
    if not document:
        print(f"PDF with key {key} not found.")
        return None
    return document["pdf_content"]  # Assuming PDF is stored in binary format

# Merge PDFs
merged_pdf = fitz.open()

for key in keys:
    pdf_data = retrieve_pdf(key)
    if pdf_data:
        doc = fitz.open("pdf", pdf_data)  # Load PDF from binary
        if len(doc) > 0:
            merged_pdf.insert_pdf(doc, from_page=0, to_page=0)  # Insert only the first page

# Save merged PDF to a byte buffer
output_pdf = BytesIO()
merged_pdf.save(output_pdf)
merged_pdf.close()

# Upload to S3
output_pdf.seek(0)
s3_filename = folder_path + "merged_output.pdf"
s3_client.upload_fileobj(output_pdf, bucket_name, s3_filename, ExtraArgs={"ContentType": "application/pdf"})

print(f"Merged PDF uploaded to s3://{bucket_name}/{s3_filename}")
